# EKF and UKF lab: nonlinear range observation

Compare a local-Jacobian EKF and scaled UKF on a simple two-state motion model with a nonlinear range observation.

In [ ]:
import sys
sys.path.insert(0, "implementations/python/state-estimation")
import numpy as np
from nonlinear_filters import ekf_filter, ukf_filter
dt = 1.0
F = np.array([[1., dt], [0., 1.]])
f = lambda x: F @ x
# Observe squared position: nonlinear and intentionally simple for the lab.
h = lambda x: np.array([x[0] ** 2])
H = lambda x: np.array([[2. * x[0], 0.]])
Q = np.diag([0.02, 0.01])
R = np.array([[0.25]])


In [ ]:
truth = np.array([2.0, 0.15])
rng = np.random.default_rng(8)
observations = []
for _ in range(30):
    truth = f(truth)
    observations.append([h(truth)[0] + rng.normal(scale=np.sqrt(R[0, 0]))])
observations = np.asarray(observations)
ekf = ekf_filter(observations, np.array([1.8, 0.0]), np.diag([0.5, 0.2]), f, h, lambda _: F, H, Q, R)
ukf = ukf_filter(observations, np.array([1.8, 0.0]), np.diag([0.5, 0.2]), f, h, Q, R, alpha=0.2)
print('EKF final state:', ekf['means'][-1])
print('UKF final state:', ukf['means'][-1])


## Interpretation
The observation h(x)=position² is non-injective: it cannot distinguish positive from negative position by itself. Initial conditions and dynamics matter. This lab is a numerical comparison, not evidence that one filter is universally superior. For bearings, wrap innovations explicitly; for geographic coordinates, choose a suitable local/geodetic state representation.